# Chapter 11: Operating, Evolving, and Retiring Agents

This notebook mirrors the chapter. Run the cells in order, from the `Chapter11`
folder. Nothing here needs an API key. The policy code is plain Python over the
files in `data/`. The memory consolidator is a real SDK agent with a structured
output, driven here by the SDK's scripted test model.

All plant, fleet, and model data is synthetic. Pinned version: openai-agents 0.24.0.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
print("openai-agents", version("openai-agents"))

openai-agents 0.24.0


## 1. Day 2, day 31, day 97: the drift view

Five signals, one decision. A configuration that is not the approved bundle holds and opens a candidate release, even when the results improved.

In [2]:
!{sys.executable} -m examples.drift_view

day 2: nothing was released
  model           DIFFERENT
    text.verbosity: approved medium, running high
  input           0.07  baseline 0.06  band 0.05  in band
  knowledge       0.02  baseline 0.02  band 0.03  in band
  outcome         0.86  baseline 0.83  band 0.02  MOVED
  cost and speed  3.36  baseline 3.17  band 0.15  MOVED
  decision: hold
  why: not the approved bundle: text.verbosity
  opens a candidate release: yes

day 31: retirement notice received
  model           as approved
  input           0.06  baseline 0.06  band 0.05  in band
  knowledge       0.03  baseline 0.02  band 0.03  in band
  outcome         0.84  baseline 0.83  band 0.02  in band
  cost and speed  3.21  baseline 3.17  band 0.15  in band
  decision: ok
  why: every signal inside its band
  opens a candidate release: no

day 97: three drafts rejected in a row at plant 2
  model           as approved
  input           0.08  baseline 0.06  band 0.05  in band
  knowledge       0.03  baseline 0.02  band 0.03 

## 2. Day 140 and day 175: a promotion case, twice

The first case fails on the one number nobody had measured: whether the reviewers still catch errors. The ceiling answers alone, and the drop rule needs no signature.

In [3]:
!{sys.executable} -m examples.promotion

day 140: planners approved 96% of drafts, and caught 8 of 26 planted errors
  maintenance-planner: rung 1 to rung 2 (approve by exception)
  gap: planted_check_catch_rate: 0.31 is below the required 0.9

day 175: caught 28 of 30 planted errors, wrongly rejected 2 of 50 clean drafts
  maintenance-planner: rung 1 to rung 2 (approve by exception)
  no gaps: goes to sponsor and security to sign

a case nobody should bring
  maintenance-planner: to rung 4 (bounded spending)
  gap: rung 4 is above this agent's ceiling of 3
  credit-decision: to rung 2, with perfect evidence
  gap: rung 2 is above this agent's ceiling of 1

day 97, morning shift at plant 2: the drop rule fired
  consecutive_denials: 3 (limit 3)
  maintenance-planner: rung 1 to rung 0 (draft only), no signature needed


## 3. A spending mandate, and what revoking it leaves behind

Revocation stops the next order. It does not undo the ones already placed.

In [4]:
!{sys.executable} -m examples.mandate_demo

consumables-buyer: up to $400 an order and $1000 a day, 3 suppliers
  on virtual card consumables-01, expires day 270
  signed by the finance controller
  PO-1  $180  NordBolt
        placed
  PO-2  $520  NordBolt
        refused: $520 is over the order cap
  PO-3  $90   Quickfix Direct
        refused: supplier is not on the mandate
  PO-4  $350  Halden Seals
        placed
  PO-5  $320  Reka Lubricants
        placed
  PO-6  $240  Reka Lubricants
        refused: the day's cap would be exceeded

mandate revoked
  PO-7  refused: mandate revoked
  still placed and not undone: PO-1, PO-4, PO-5 ($850)
  each needs a person: cancel with the supplier, or let it complete


## 4. Day 180: the memory hygiene pass

Expire, supersede, merge, verify, scored before and after. The merge is accepted on checks that do not depend on the model's own confidence.

In [5]:
!{sys.executable} -m examples.hygiene

before: asked about the feed pump, the agent reads m01
  Feed pump P3 is a KS-65, serviced every 120 days, 40 Nm on the housing.


  expired      2
  superseded   2
  merged       1
  quarantined  1
  flagged      0
  unconfirmed  0
recall      70% before, 100% after
forgetting  25% before, 100% after
after: asked about the feed pump, the agent reads m02
  Feed pump P3 was replaced with a KS-80: 90 days, 55 Nm.
waiting for a person: m09 (no source on record)


## 5. Lineage: a poisoned source and an erasure request

One walk answers both questions: what did this source produce, and where does every copy live.

In [6]:
!{sys.executable} -m examples.lineage_demo

source email-7731 (contractor, unverified) produced 3 memories
  m20  fact     email-7731   quarantined
  m21  summary  from m20     quarantined
  m22  note     from m21     quarantined
  what the agent can retrieve about plant 2 lockout now: None

erasure request: sources session-0102, session-0188
  copies found: m30, m31, m33, m32
  deleted from the store and the index: m30, m31, m32
  deleted from the archive of originals: m30, m31
  kept and restricted: m33
    basis: safety inspection records are kept for the period the law requires
  backup weekly-22 still holds m30, m31, m32: marked
  backup weekly-27 still holds m30, m31, m32: marked
restoring weekly-22 later: m30, m31, m32 are not brought back
  the retained record is still hidden from retrieval: True


## 6. Day 240: fifteen agents

The listing check, the reconciliation with two vendor registries, and a contract change that only registered consumers were protected from.

In [7]:
!{sys.executable} -m examples.fleet_demo

on the register: 15 agents, 11 found by the sweep
  energy-report      cannot be listed: 5 gaps
  invoice-matcher    cannot be listed: 5 gaps
  quality-notes      cannot be listed: 5 gaps
  onboarding-helper  cannot be listed: 4 gaps

nightly reconciliation with two vendor registries
  spares-forecast: in cloud-a, and nobody sponsors it
  calibration-reminder: in cloud-a, and nobody sponsors it
  invoice-matcher: in cloud-a, and nobody sponsors it
  quality-notes: in cloud-a, and nobody sponsors it
  onboarding-helper: in cloud-b, and nobody sponsors it
  purchasing-advisor: on the vendor's platform, which no registry covers
  shift-report: we say cloud-b, and cloud-b does not list it
  energy-report: on an engineer's laptop, which no registry covers

parts-availability moves from contract v2 to v3
  maintenance-planner  consumer test: BREAKS, it reads lead_time_days
  spares-forecast      consumer test: passes
  shift-report         not registered, so no test: reads lead_time_days


## 7. Day 300 onward: the agent proposes, the gate decides

Three proposed fixes and two stop drills on two clocks.

In [8]:
!{sys.executable} -m examples.improvement

fix 1, a tool description: replay 3 of 3 now pass
  golden 0.97 to 0.97, held-out 0.94 to 0.95, judge 0.81 to 0.84
  passes the gate, approved by the planner lead, plant 1
fix 2, a prompt change: replay 5 of 5 now pass
  golden 0.97 to 0.97, held-out 0.95 to 0.95, judge 0.84 to 0.91
  held: the judge likes it more and the held-out set does not
  held: drafts got longer and planners edit them just as much
fix 3, a looser criterion: replay 2 of 2 now pass
  golden 0.97 to 0.99, held-out 0.95 to 0.95, judge 0.84 to 0.84
  held: evals/criteria.yaml: the loop may read this and never write it
  held: needs an approver who is not the proposer

first drill, day 315
  damage clock: volume deleted 4 seconds after the request
    recoverable for 48 hours, because deletes are soft
  paging clock: alert went to a chat channel read in working hours
    a person responded after 3 h 12 min
    the 30-minute rule would have paused the agent first
second drill, day 320
  damage clock: volume deleted 4 s

## 8. Retiring an agent

The script refuses while a credential is live, then completes and writes the record.

In [9]:
!{sys.executable} -m examples.retirement

first attempt refused: credentials still live: ['legacy-gw-0098']
  status on the register: deprecated
  that token came from the old gateway, and is revoked there by hand
second attempt completed
  final bundle      alert-sorter@7.2
  successor         maintenance-planner
  callers moved     plant-1-dashboard, shift-report
  not on the list   shift-report
  tokens revoked    3, none live
  archive           bundle, golden_suite, eval_history, traces, decisions
  kept until        seven years after retirement
  signed by         the plant 1 manager
  status            retired


## 9. Day 365: one year on one board

The rung by day, the drift record, and the ten events of the year.

In [10]:
!{sys.executable} -m examples.year_board

maintenance-planner, day 365
  rung 3 of a ceiling of 3 (audited autonomy)
  cost per approved order $3.17 at launch, $2.87 now (-9.5%)
  drift signals tripped on 22 days, 19 confirmed

  rung by day
    day   0 to  97  rung 1  approve every action
    day  97 to  99  rung 0  draft only
    day  99 to 175  rung 1  approve every action
    day 175 to 322  rung 2  approve by exception
    day 322 to 365  rung 3  audited autonomy

  the year in ten events
     1  day   2  upstream model change caught
     2  day  31  successor model through the gate
     3  day  58  failover drill
     4  day  97  automatic drop to rung 0
     5  day 175  promoted to rung 2
     6  day 180  first memory hygiene pass
     7  day 240  fleet sweep finds eleven more
     8  day 300  first self-proposed fix merged
     9  day 322  promoted to rung 3
    10  day 365  new sponsor signs


## 10. Try it yourself

Change one number and rerun a step. Raise `planted_check_catch_rate` in
`data/evidence.yaml`, move a band in `data/readings.yaml`, or delete a consumer
from `data/fleet.yaml` and watch the contract check stop protecting it.

In [11]:
from chapter11.autonomy import promotion_gaps
from chapter11.settings import load

rule = load("ladder")["rung_1_to_2"]
evidence = {**load("evidence")["day_140"]["evidence"]}
print(promotion_gaps(evidence, rule, target=2, ceiling=3))
evidence["planted_check_catch_rate"] = 0.95
print(promotion_gaps(evidence, rule, target=2, ceiling=3))

['planted_check_catch_rate: 0.31 is below the required 0.9']
[]
